# Kimi API 综合测试

本 notebook 用于测试 Kimi API 的所有功能。

In [ ]:
from openai import OpenAI
import os
import base64
import json
from dotenv import load_dotenv

load_dotenv(dotenv_path='../../.env')

api_key = os.getenv("VITE_KIMI_API_KEY") or os.getenv("KIMI_API_KEY")
base_url = os.getenv("VITE_KIMI_BASE_URL", "https://api.moonshot.cn/v1")

if not api_key:
    raise ValueError("❌ 未找到 API Key")

client = OpenAI(api_key=api_key, base_url=base_url)
print("✅ Kimi 客户端初始化成功")
print(f"📍 Base URL: {base_url}")

## 测试 1: 基础对话

In [ ]:
def test_basic_chat():
    print("=== 测试基础对话 ===")
    models = ["kimi-k2-turbo-preview", "kimi-k2.5"]
    
    for model in models:
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": "Hello"}],
            )
            print(f"✅ {model}: {response.choices[0].message.content[:50]}...")
        except Exception as e:
            print(f"❌ {model}: {e}")

test_basic_chat()

## 测试 2: 流式输出

In [ ]:
def test_streaming():
    print("\n=== 测试流式输出 ===")
    response = client.chat.completions.create(
        model="kimi-k2-turbo-preview",
        messages=[{"role": "user", "content": "Hello"}],
        stream=True,
    )
    
    content = ""
    for chunk in response:
        delta = chunk.choices[0].delta.content
        if delta:
            content += delta
    
    print(f"✅ 流式输出: {content[:50]}...")

test_streaming()

## 测试 3: 思考模式

In [ ]:
def test_thinking():
    print("\n=== 测试思考模式 ===")
    
    # Test kimi-k2-thinking
    try:
        response = client.chat.completions.create(
            model="kimi-k2-thinking",
            messages=[{"role": "user", "content": "解方程 2x + 5 = 13"}],
        )
        msg = response.choices[0].message
        has_reasoning = hasattr(msg, 'reasoning_content')
        print(f"✅ kimi-k2-thinking: 有思考过程={has_reasoning}")
    except Exception as e:
        print(f"❌ kimi-k2-thinking: {e}")
    
    # Test kimi-k2.5 with thinking
    try:
        response = client.chat.completions.create(
            model="kimi-k2.5",
            messages=[{"role": "user", "content": "解方程 2x + 5 = 13"}],
            thinking={"type": "enabled"},
        )
        msg = response.choices[0].message
        has_reasoning = hasattr(msg, 'reasoning_content')
        print(f"✅ kimi-k2.5 (thinking): 有思考过程={has_reasoning}")
    except Exception as e:
        print(f"❌ kimi-k2.5 (thinking): {e}")

test_thinking()

## 测试 4: 工具调用

In [ ]:
def test_tool_calling():
    print("\n=== 测试工具调用 ===")
    
    tools = [{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取天气",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"]
            }
        }
    }]
    
    try:
        response = client.chat.completions.create(
            model="kimi-k2-turbo-preview",
            messages=[{"role": "user", "content": "北京天气怎么样？"}],
            tools=tools,
        )
        msg = response.choices[0].message
        has_tool_calls = msg.tool_calls is not None
        print(f"✅ 工具调用: 触发工具={has_tool_calls}")
    except Exception as e:
        print(f"❌ 工具调用: {e}")

test_tool_calling()

## 测试 5: JSON 模式

In [ ]:
def test_json_mode():
    print("\n=== 测试 JSON 模式 ===")
    
    try:
        response = client.chat.completions.create(
            model="kimi-k2-turbo-preview",
            messages=[
                {"role": "user", "content": "输出 JSON: {\"name\": \"test\", \"value\": 123}"}
            ],
            response_format={"type": "json_object"},
        )
        content = response.choices[0].message.content
        data = json.loads(content)
        print(f"✅ JSON 模式: 解析成功={isinstance(data, dict)}")
    except Exception as e:
        print(f"❌ JSON 模式: {e}")

test_json_mode()

## 测试 6: 模型列表

In [ ]:
def test_list_models():
    print("\n=== 测试模型列表 ===")
    
    try:
        models = client.models.list()
        model_ids = [m.id for m in models.data]
        print(f"✅ 可用模型 ({len(model_ids)} 个):")
        for model_id in sorted(model_ids):
            print(f"  - {model_id}")
    except Exception as e:
        print(f"❌ 模型列表: {e}")

test_list_models()

## 运行所有测试

In [ ]:
def run_all_tests():
    print("开始运行所有测试...\n")
    
    test_basic_chat()
    test_streaming()
    test_thinking()
    test_tool_calling()
    test_json_mode()
    test_list_models()
    
    print("\n✅ 所有测试完成")

run_all_tests()